# Tratamento e análise de dados — Escola de Artes Marciais

Projeto de dados com Python e Pandas. A base é fictícia e foi preparada para testar as regras de limpeza, cobrança e frequência. Os resultados não são dados reais da escola.

Data de referência: **02/11/2026**. Multa de **2%** e frequência mínima de **75%** são parâmetros de teste, ainda não confirmados com a escola.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

arquivo = Path('Base_Escola.csv')
data_teste = pd.Timestamp('2026-11-02')
multa_teste = 0.02
frequencia_minima = 75

if not arquivo.exists():
    raise FileNotFoundError('Coloque Base_Escola.csv na mesma pasta do notebook.')

In [2]:
original = pd.read_csv(arquivo, dtype=str, keep_default_na=False, encoding='utf-8-sig')
colunas = [
    'id_aluno', 'nome', 'modalidade', 'turma', 'faixa_atual',
    'valor_mensalidade', 'vencimento', 'data_pagamento',
    'quantidade_aulas', 'presencas'
]
assert set(colunas).issubset(original.columns), 'O CSV está sem alguma coluna necessária.'

print('Linhas da base:', len(original))
display(original.head())


Linhas da base: 1652


,id_aluno,nome,modalidade,turma,faixa_atual,valor_mensalidade,vencimento,data_pagamento,quantidade_aulas,presencas
0,503,Fernanda Fonseca,Jiu-jitsu,T03,Cinza,210.00,2026-10-12,2026-10-18,17,11
1,506,Otávio Andrade,Jiu-jitsu,T01,Amarela,100.00,2026-11-28,,14,13
2,507,André Araújo,Jiu-jitsu,T16,Verde,180.00,17/10/2026,2026-10-16,14,10
3,508,Samuel Xavier,Muay Thai,T18,Cinza,130.00,2026-11-22,2026-10-31,14,11
4,511,Mariana Monteiro,Judô,,Amarela,150.00,2026-10-10,,17,13


In [3]:
modalidades = {
    'jiu-jitsu': 'Jiu-jitsu', 'jiu jitsu': 'Jiu-jitsu',
    'muay thai': 'Muay Thai', 'judô': 'Judô', 'judo': 'Judô',
    'karatê': 'Karatê', 'karate': 'Karatê',
}

dados = original[colunas].copy()
dados['linha'] = dados.index + 2
dados['id_aluno'] = dados['id_aluno'].str.strip()
dados['nome'] = dados['nome'].str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
dados['modalidade'] = (dados['modalidade'].str.strip()
                       .str.replace(r'\s+', ' ', regex=True)
                       .str.lower().map(modalidades))
dados['turma'] = dados['turma'].str.strip().str.upper()
dados['faixa_atual'] = (dados['faixa_atual'].str.strip()
                        .str.replace(r'\s+', ' ', regex=True).str.title())

print('Campos de texto padronizados.')
display(dados[['nome', 'modalidade', 'turma', 'faixa_atual']].head())


Campos de texto padronizados.


,nome,modalidade,turma,faixa_atual
0,Fernanda Fonseca,Jiu-jitsu,T03,Cinza
1,Otávio Andrade,Jiu-jitsu,T01,Amarela
2,André Araújo,Jiu-jitsu,T16,Verde
3,Samuel Xavier,Muay Thai,T18,Cinza
4,Mariana Monteiro,Judô,,Amarela


In [4]:
texto_valor = (dados['valor_mensalidade'].str.strip()
              .str.replace('R$', '', regex=False)
              .str.replace(' ', '', regex=False)
              .str.replace(',', '.', regex=False))
valor = pd.to_numeric(texto_valor, errors='coerce')
texto_aulas = dados['quantidade_aulas'].str.strip()
texto_presencas = dados['presencas'].str.strip()
aulas = pd.to_numeric(texto_aulas, errors='coerce')
presencas = pd.to_numeric(texto_presencas, errors='coerce')

def ler_datas(serie):
    formato_ano = pd.to_datetime(serie, format='%Y-%m-%d', errors='coerce')
    formato_br = pd.to_datetime(serie, format='%d/%m/%Y', errors='coerce')
    return formato_ano.fillna(formato_br)

vencimento = ler_datas(dados['vencimento'])
texto_pagamento = dados['data_pagamento'].str.strip()
pagamento = ler_datas(texto_pagamento)

print('Valores e datas convertidos.')


Valores e datas convertidos.


In [5]:
motivos = pd.Series('', index=dados.index, dtype='str')

def marcar_erro(condicao, motivo):
    motivos.loc[motivos.eq('') & condicao.fillna(False)] = motivo

marcar_erro(dados['id_aluno'].eq('') | dados['nome'].eq('') |
            dados['turma'].eq('') | dados['faixa_atual'].eq(''),
            'cadastro com campo obrigatório vazio')
marcar_erro(dados['modalidade'].isna(), 'modalidade não reconhecida')
marcar_erro(valor.isna() | aulas.isna() | presencas.isna() |
            ~texto_aulas.str.fullmatch(r'[+-]?\d+') |
            ~texto_presencas.str.fullmatch(r'[+-]?\d+'), 'número inválido')
marcar_erro(valor.le(0), 'valor da mensalidade inválido')
marcar_erro(aulas.le(0) | presencas.lt(0) | presencas.gt(aulas),
            'quantidade de aulas ou presenças inválida')
marcar_erro(vencimento.isna() | (texto_pagamento.ne('') & pagamento.isna()),
            'data de vencimento ou pagamento inválida')
marcar_erro(pagamento.gt(data_teste), 'pagamento com data futura')

candidatos = dados.loc[motivos.eq('')]
repetidos = candidatos.duplicated(subset='id_aluno', keep='first')
motivos.loc[candidatos.index[repetidos]] = 'identificador repetido'

validos = motivos.eq('')
rejeitados = pd.DataFrame({
    'linha': dados.loc[~validos, 'linha'],
    'id_aluno': original.loc[~validos, 'id_aluno'],
    'motivo': motivos.loc[~validos]
})

print('Registros aproveitados:', int(validos.sum()))
print('Registros rejeitados:', len(rejeitados))
display(rejeitados.head(8))


Registros aproveitados: 1484
Registros rejeitados: 168


,linha,id_aluno,motivo
4,6,511,cadastro com campo obrigatório vazio
16,18,546,pagamento com data futura
25,27,579,cadastro com campo obrigatório vazio
32,34,601,modalidade não reconhecida
41,43,637,cadastro com campo obrigatório vazio
46,48,658,cadastro com campo obrigatório vazio
48,50,,cadastro com campo obrigatório vazio
62,64,704,modalidade não reconhecida


In [6]:
tratados = dados.loc[validos, colunas].copy()
tratados['valor_mensalidade'] = valor.loc[validos].map(lambda n: f'{n:.2f}')
tratados['vencimento'] = vencimento.loc[validos].dt.strftime('%Y-%m-%d')
tratados['data_pagamento'] = pagamento.loc[validos].dt.strftime('%Y-%m-%d').fillna('')
tratados['quantidade_aulas'] = aulas.loc[validos].astype(int)
tratados['presencas'] = presencas.loc[validos].astype(int)

tratados['situacao_pagamento'] = 'Em aberto'
tratados.loc[vencimento.loc[validos].lt(data_teste), 'situacao_pagamento'] = 'Atrasado'
tratados.loc[pagamento.loc[validos].notna(), 'situacao_pagamento'] = 'Pago'
tratados['multa_exemplo'] = '0.00'
atrasados = tratados['situacao_pagamento'].eq('Atrasado')
tratados.loc[atrasados, 'multa_exemplo'] = (
    valor.loc[tratados.index[atrasados]] * multa_teste
).map(lambda n: f'{n:.2f}')

frequencia_calculada = 100 * tratados['presencas'] / tratados['quantidade_aulas']
tratados['frequencia_percentual'] = frequencia_calculada.round(1)
tratados['frequencia_minima_atingida'] = (
    frequencia_calculada >= frequencia_minima
).map({True: 'Sim', False: 'Não'})

display(tratados.head(6))


,id_aluno,nome,modalidade,turma,faixa_atual,valor_mensalidade,vencimento,data_pagamento,quantidade_aulas,presencas,situacao_pagamento,multa_exemplo,frequencia_percentual,frequencia_minima_atingida
0,503,Fernanda Fonseca,Jiu-jitsu,T03,Cinza,210.00,2026-10-12,2026-10-18,17,11,Pago,0.00,64.7,Não
1,506,Otávio Andrade,Jiu-jitsu,T01,Amarela,100.00,2026-11-28,,14,13,Em aberto,0.00,92.9,Sim
2,507,André Araújo,Jiu-jitsu,T16,Verde,180.00,2026-10-17,2026-10-16,14,10,Pago,0.00,71.4,Não
3,508,Samuel Xavier,Muay Thai,T18,Cinza,130.00,2026-11-22,2026-10-31,14,11,Pago,0.00,78.6,Sim
5,513,Elaine Dias,Muay Thai,T01,Amarela,100.00,2026-11-10,,15,13,Em aberto,0.00,86.7,Sim
6,514,Diego Oliveira,Muay Thai,T01,Verde,125.00,2026-11-28,2026-10-30,10,5,Pago,0.00,50.0,Não


In [7]:
lista_correcoes = []
campos_conferidos = [
    'nome', 'modalidade', 'turma', 'faixa_atual',
    'valor_mensalidade', 'vencimento', 'data_pagamento'
]

for posicao, campo in enumerate(campos_conferidos):
    alterou = original.loc[validos, campo].ne(tratados[campo])
    indices = tratados.index[alterou]
    if len(indices) > 0:
        lista_correcoes.append(pd.DataFrame({
            'linha': indices + 2,
            'id_aluno': tratados.loc[indices, 'id_aluno'].to_numpy(),
            'campo': campo,
            'antes': original.loc[indices, campo].to_numpy(),
            'depois': tratados.loc[indices, campo].to_numpy(),
            '_ordem': posicao,
        }))

if lista_correcoes:
    correcoes = (pd.concat(lista_correcoes, ignore_index=True)
                .sort_values(['linha', '_ordem']).drop(columns='_ordem'))
else:
    correcoes = pd.DataFrame(columns=['linha', 'id_aluno', 'campo', 'antes', 'depois'])

print('Registros que tiveram correção:', correcoes['linha'].nunique())
print('Campos corrigidos:', len(correcoes))
display(correcoes.head(10))


Registros que tiveram correção: 878
Campos corrigidos: 1259


,linha,id_aluno,campo,antes,depois
1044,4,507,vencimento,17/10/2026,2026-10-17
262,7,513,modalidade,muay thai,Muay Thai
486,7,513,turma,t01,T01
674,7,513,faixa_atual,amarela,Amarela
675,8,514,faixa_atual,verde,Verde
803,9,515,valor_mensalidade,"R$ 130,00",130.00
1045,9,515,vencimento,22/10/2026,2026-10-22
0,14,536,nome,LEONARDO COSTA,Leonardo Costa
804,14,536,valor_mensalidade,"R$ 125,00",125.00
1,15,540,nome,CÉSAR ALMEIDA BATISTA,César Almeida Batista


In [8]:
turmas = (tratados[['modalidade', 'turma']].drop_duplicates()
          .sort_values(['modalidade', 'turma']).reset_index(drop=True))
turmas.insert(0, 'id_turma', ['TUR-' + str(n).zfill(3) for n in range(1, len(turmas)+1)])

planos = (tratados[['valor_mensalidade']].drop_duplicates()
          .sort_values('valor_mensalidade', key=lambda s: s.astype(float)).reset_index(drop=True))
planos.insert(0, 'id_plano', ['PLA-' + str(n).zfill(3) for n in range(1, len(planos)+1)])

alunos = (tratados[['id_aluno', 'nome', 'modalidade', 'turma',
                    'faixa_atual', 'valor_mensalidade']]
          .merge(turmas, on=['modalidade','turma'], how='left', validate='many_to_one')
          .merge(planos, on='valor_mensalidade', how='left', validate='many_to_one'))
alunos = alunos[['id_aluno', 'nome', 'modalidade', 'id_turma', 'id_plano', 'faixa_atual']]

print('Alunos:', len(alunos), '| Turmas:', len(turmas), '| Valores de plano:', len(planos))
display(alunos.head())


Alunos: 1484 | Turmas: 72 | Valores de plano: 16


,id_aluno,nome,modalidade,id_turma,id_plano,faixa_atual
0,503,Fernanda Fonseca,Jiu-jitsu,TUR-003,PLA-016,Cinza
1,506,Otávio Andrade,Jiu-jitsu,TUR-001,PLA-003,Amarela
2,507,André Araújo,Jiu-jitsu,TUR-016,PLA-014,Verde
3,508,Samuel Xavier,Muay Thai,TUR-072,PLA-008,Cinza
4,513,Elaine Dias,Muay Thai,TUR-055,PLA-003,Amarela


In [9]:
pagamentos = tratados[['id_aluno', 'valor_mensalidade', 'vencimento',
                        'data_pagamento', 'situacao_pagamento']].copy()
pagamentos.insert(0, 'id_pagamento', [f'PAG-{n:05d}' for n in range(1, len(pagamentos)+1)])
pagamentos['mes_referencia'] = pagamentos['vencimento'].str[:7]
pagamentos['dias_atraso'] = 0

indice = pagamentos.index
em_aberto_atrasado = pagamentos['situacao_pagamento'].eq('Atrasado')
pago_depois = (pagamentos['situacao_pagamento'].eq('Pago') &
               pagamento.loc[indice].gt(vencimento.loc[indice]))
pagamentos.loc[em_aberto_atrasado, 'dias_atraso'] = (
    data_teste - vencimento.loc[indice[em_aberto_atrasado]]
).dt.days.clip(lower=0).astype(int)
pagamentos.loc[pago_depois, 'dias_atraso'] = (
    pagamento.loc[indice[pago_depois]] - vencimento.loc[indice[pago_depois]]
).dt.days.astype(int)
pagamentos['pago_apos_vencimento'] = pago_depois.map({True: 'Sim', False: 'Não'})

pagamentos['multa_estimada'] = 0.0
pagamentos.loc[em_aberto_atrasado, 'multa_estimada'] = (
    pagamentos.loc[em_aberto_atrasado, 'valor_mensalidade'].astype(float) * multa_teste
).round(2)
pagamentos['valor_estimado'] = (
    pagamentos['valor_mensalidade'].astype(float) + pagamentos['multa_estimada']
).round(2)
pagamentos['confirmacao_manual'] = 'Não'

print(pagamentos['situacao_pagamento'].value_counts().to_string())
display(pagamentos.head())


situacao_pagamento
Pago         721
Atrasado     387
Em aberto    376


,id_pagamento,id_aluno,valor_mensalidade,vencimento,data_pagamento,situacao_pagamento,mes_referencia,dias_atraso,pago_apos_vencimento,multa_estimada,valor_estimado,confirmacao_manual
0,PAG-00001,503,210.00,2026-10-12,2026-10-18,Pago,2026-10,6,Sim,0.0,210.0,Não
1,PAG-00002,506,100.00,2026-11-28,,Em aberto,2026-11,0,Não,0.0,100.0,Não
2,PAG-00003,507,180.00,2026-10-17,2026-10-16,Pago,2026-10,0,Não,0.0,180.0,Não
3,PAG-00004,508,130.00,2026-11-22,2026-10-31,Pago,2026-11,0,Não,0.0,130.0,Não
5,PAG-00005,513,100.00,2026-11-10,,Em aberto,2026-11,0,Não,0.0,100.0,Não


In [10]:
frequencias = tratados[['id_aluno', 'quantidade_aulas', 'presencas',
                         'frequencia_percentual']].copy()
graduacoes = tratados[['id_aluno', 'faixa_atual', 'frequencia_percentual']].copy()
graduacoes['atingiu_frequencia_minima'] = tratados['frequencia_minima_atingida']
graduacoes['situacao'] = graduacoes['atingiu_frequencia_minima'].map({
    'Sim': 'Pode ser avaliado pelo professor',
    'Não': 'Ainda não atingiu a frequência mínima'
})

print(graduacoes['atingiu_frequencia_minima'].value_counts().to_string())
display(graduacoes.head())


atingiu_frequencia_minima
Não    743
Sim    741


,id_aluno,faixa_atual,frequencia_percentual,atingiu_frequencia_minima,situacao
0,503,Cinza,64.7,Não,Ainda não atingiu a frequência mínima
1,506,Amarela,92.9,Sim,Pode ser avaliado pelo professor
2,507,Verde,71.4,Não,Ainda não atingiu a frequência mínima
3,508,Cinza,78.6,Sim,Pode ser avaliado pelo professor
5,513,Amarela,86.7,Sim,Pode ser avaliado pelo professor


In [11]:
def confirmar_pagamento(tabela, id_aluno, data_confirmada, data_limite=data_teste):
    alterada = tabela.copy()
    chave = str(id_aluno)
    mask = alterada['id_aluno'].astype(str).eq(chave)
    if mask.sum() != 1:
        raise ValueError('Aluno não encontrado ou identificado mais de uma vez')
    if alterada.loc[mask, 'situacao_pagamento'].iloc[0] == 'Pago':
        raise ValueError('Pagamento já registrado para este aluno')
    dia = pd.to_datetime(data_confirmada, format='%Y-%m-%d', errors='raise')
    if dia > data_limite:
        raise ValueError('A data não pode ser maior que a data do teste')
    alterada.loc[mask, 'data_pagamento'] = dia.strftime('%Y-%m-%d')
    alterada.loc[mask, 'situacao_pagamento'] = 'Pago'
    alterada.loc[mask, 'confirmacao_manual'] = 'Sim'
    venc = pd.to_datetime(alterada.loc[mask, 'vencimento'].iloc[0])
    atraso = max((dia - venc).days, 0)
    alterada.loc[mask, 'dias_atraso'] = atraso
    alterada.loc[mask, 'pago_apos_vencimento'] = 'Sim' if atraso > 0 else 'Não'
    alterada.loc[mask, 'multa_estimada'] = 0.0
    alterada.loc[mask, 'valor_estimado'] = alterada.loc[mask, 'valor_mensalidade'].astype(float)
    return alterada

exemplo_id = pagamentos.loc[pagamentos['situacao_pagamento'].eq('Atrasado'), 'id_aluno'].iloc[0]
exemplo_baixa = confirmar_pagamento(pagamentos, exemplo_id, '2026-11-02')
print('Antes:', pagamentos.loc[pagamentos['id_aluno'].eq(exemplo_id), 'situacao_pagamento'].iloc[0])
print('Depois:', exemplo_baixa.loc[exemplo_baixa['id_aluno'].eq(exemplo_id), 'situacao_pagamento'].iloc[0])


Antes: Atrasado
Depois: Pago


In [12]:
verificacoes = {
    'separação de registros': len(original) == len(tratados) + len(rejeitados),
    'IDs únicos': tratados['id_aluno'].is_unique,
    'quantidade nas tabelas': len(alunos) == len(pagamentos) == len(frequencias) == len(graduacoes),
    'situações de pagamento': set(pagamentos['situacao_pagamento']) <= {'Pago', 'Em aberto', 'Atrasado'},
    'pagamentos com data': pagamentos.loc[pagamentos['situacao_pagamento'].eq('Pago'), 'data_pagamento'].ne('').all(),
    'pendências sem data': pagamentos.loc[pagamentos['situacao_pagamento'].ne('Pago'), 'data_pagamento'].eq('').all(),
    'turmas e planos associados': alunos[['id_turma', 'id_plano']].notna().all().all(),
    'frequência entre 0% e 100%': frequencias['frequencia_percentual'].between(0, 100).all(),
    'sem multa nas situações regulares': (pagamentos.loc[pagamentos['situacao_pagamento'].ne('Atrasado'), 'multa_estimada'] == 0).all(),
    'dias de atraso positivos': (pagamentos.loc[pagamentos['situacao_pagamento'].eq('Atrasado'), 'dias_atraso'] > 0).all(),
    'correções identificadas': len(correcoes) >= correcoes['linha'].nunique(),
    'confirmação manual demonstrada': exemplo_baixa.loc[exemplo_baixa['id_aluno'].eq(exemplo_id), 'confirmacao_manual'].iloc[0] == 'Sim',
}
for descricao, passou in verificacoes.items():
    assert bool(passou), f'Falha na verificação: {descricao}'
print(f'{len(verificacoes)} verificações concluídas.')

12 verificações concluídas.


In [13]:
print('Modalidades:')
print(tratados['modalidade'].value_counts().to_string())
print('\nPagamentos:')
print(pagamentos['situacao_pagamento'].value_counts().to_string())
print('\nFrequência para avaliação:')
print(graduacoes['atingiu_frequencia_minima'].value_counts().to_string())
print('\nMotivos de rejeição:')
print(rejeitados['motivo'].value_counts().to_string())

Modalidades:
modalidade
Jiu-jitsu    672
Muay Thai    445
Judô         247
Karatê       120

Pagamentos:
situacao_pagamento
Pago         721
Atrasado     387
Em aberto    376

Frequência para avaliação:
atingiu_frequencia_minima
Não    743
Sim    741

Motivos de rejeição:
motivo
cadastro com campo obrigatório vazio         51
quantidade de aulas ou presenças inválida    29
identificador repetido                       21
número inválido                              17
pagamento com data futura                    15
modalidade não reconhecida                   13
valor da mensalidade inválido                12
data de vencimento ou pagamento inválida     10


In [14]:
saida_final = Path('.')

arquivos_saida = {
    'dados_tratados_final.csv': tratados,
    'correcoes_final.csv': correcoes,
    'rejeitados_final.csv': rejeitados,
    'alunos_final.csv': alunos,
    'turmas_final.csv': turmas,
    'planos_final.csv': planos,
    'pagamentos_final.csv': pagamentos,
    'frequencias_final.csv': frequencias,
    'graduacoes_final.csv': graduacoes,
    'baixa_manual_exemplo.csv': exemplo_baixa.loc[exemplo_baixa['id_aluno'].eq(exemplo_id)].copy(),
}
for nome_arquivo, tabela in arquivos_saida.items():
    tabela.to_csv(saida_final / nome_arquivo, index=False, encoding='utf-8-sig')

resumo_final = (f'Data de referência: {data_teste.date()}\n'
    f'Registros lidos: {len(original)}\n'
    f'Registros aproveitados: {len(tratados)}\n'
    f'Registros rejeitados: {len(rejeitados)}\n'
    f'Registros com correção: {correcoes["linha"].nunique()}\n'
    f'Campos corrigidos: {len(correcoes)}\n'
    f'Pagos: {(pagamentos["situacao_pagamento"] == "Pago").sum()}\n'
    f'Em aberto: {(pagamentos["situacao_pagamento"] == "Em aberto").sum()}\n'
    f'Atrasados: {(pagamentos["situacao_pagamento"] == "Atrasado").sum()}\n'
    f'Frequência mínima de teste atingida: {(graduacoes["atingiu_frequencia_minima"] == "Sim").sum()}\n'
    f'Verificações: {len(verificacoes)}\n'
    'Base fictícia. Multa de 2% e frequência mínima de 75% usadas apenas nos testes.\n'
)
(saida_final / 'resumo_final.txt').write_text(resumo_final, encoding='utf-8')
print(resumo_final)
print('Arquivos finais salvos na pasta do projeto.')

Data de referência: 2026-11-02
Registros lidos: 1652
Registros aproveitados: 1484
Registros rejeitados: 168
Registros com correção: 878
Campos corrigidos: 1259
Pagos: 721
Em aberto: 376
Atrasados: 387
Frequência mínima de teste atingida: 741
Verificações: 12
Base fictícia. Multa de 2% e frequência mínima de 75% usadas apenas nos testes.

Arquivos finais salvos na pasta do projeto.
